In [ ]:
import modeling_selfcheck_apiprompt
import torch
import spacy
import warnings
from openai import OpenAI
import os
warnings.filterwarnings("ignore")

selfcheck_apiprompt_solar_pro = modeling_selfcheck_apiprompt.SelfCheckAPIPrompt(client_type="openai", 
                                                                      client=OpenAI(api_key=os.environ["UPSTAGE_API_KEY"], 
                                                                                    base_url="https://api.upstage.ai/v1/solar"), 
                                                                        model="solar-pro")

In [ ]:
import json

with open("data/dataset_v3.json", "r") as f:
    content = f.read()
dataset = json.loads(content)
print("The lenght of the dataset: {}".format(len(dataset)))
print(dataset[0].keys())

In [ ]:
import numpy as np

label_mapping = {
    'accurate': 0.0,
    'minor_inaccurate': 0.5,
    'major_inaccurate': 1.0,
}

human_label_detect_False   = {}
human_label_detect_True    = {}
human_label_detect_False_h = {}

for i_ in range(len(dataset)):
    dataset_i = dataset[i_]
    idx = dataset_i["wiki_bio_test_idx"]
    
    raw_label = np.array([label_mapping[x] for x in dataset_i['annotation']])
    human_label_detect_False[idx] = (raw_label > 0.499).astype(np.int32).tolist()
    human_label_detect_True[idx] = (raw_label < 0.499).astype(np.int32).tolist()
    average_score = np.mean(raw_label)
    if (average_score < 0.99):
        human_label_detect_False_h[idx] = (raw_label > 0.99).astype(np.int32).tolist()

In [ ]:
print("Length of False:", len(human_label_detect_False))
print("Length of True:", len(human_label_detect_True)) 
print("Length of False_h:", len(human_label_detect_False_h))

In [ ]:
from tqdm import tqdm

self_check_scores_solar_pro = {}

for i in tqdm(range(len(dataset))):
    x = dataset[i]
    idx = dataset[i]['wiki_bio_test_idx']x

    self_check_scores_solar_pro[idx] = selfcheck_apiprompt_solar_pro.predict(
        sentences = x['gpt3_sentences'],           
        sampled_passages = x['gpt3_text_samples'],
        verbose=True
    )

In [ ]:
selfcheck_scores_promptapi_solar_pro = {}

for idx in self_check_scores_solar_pro:
  scores = self_check_scores_solar_pro[idx]
  selfcheck_scores_promptapi_solar_pro[idx] = [score for score in scores]

with open("data/selfcheck_scores_promptapi_solar_pro.json", "w") as outfile:
    json.dump(selfcheck_scores_promptapi_solar_pro, outfile)